# Ultrasound classification under acquisition shift (public data)

Train two 160-tree classifiers on pixel intensities. Compare clean training with training on clean and randomly degraded images. Evaluate on the same clean and perturbed test cases. The perturbation changes gain, blur and noise. BreastMNIST has 546 training and 156 official test ultrasound images at 28 × 28 resolution. The positive label combines normal and benign cases. All three fits share one test set. The acquisition shift is simulated.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'ultrasound-acquisition-shift-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0']],check=True)

In [ ]:
(root / 'prepare.py').write_text('from pathlib import Path\nimport hashlib\nimport os\nimport urllib.request\nimport zipfile\n\nroot = Path(os.environ.get("PORTFOLIO_DATA", Path(__file__).resolve().parent / "data"))\npath = root / \'medmnist/breastmnist.npz\'\npath.parent.mkdir(parents=True, exist_ok=True)\nif not path.exists():\n    with urllib.request.urlopen(\'https://zenodo.org/records/10519652/files/breastmnist.npz?download=1\',timeout=120) as response:\n        payload = response.read()\n    if hashlib.sha256(payload).hexdigest() != \'3a22d2a4625e7dd8903f0f59030e81a3d0c222bece2e5e10927456cb50efdec1\':\n        raise ValueError("Downloaded release checksum mismatch")\n    path.write_bytes(payload)\nif hashlib.sha256(path.read_bytes()).hexdigest() != \'3a22d2a4625e7dd8903f0f59030e81a3d0c222bece2e5e10927456cb50efdec1\':\n    raise ValueError("Local dataset checksum mismatch")\n',encoding='utf8')
subprocess.run([sys.executable,'prepare.py'],cwd=root,check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom scipy import ndimage as ndi\nfrom sklearn.ensemble import ExtraTreesClassifier\nfrom sklearn.metrics import balanced_accuracy_score, roc_auc_score\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", str(ROOT / "data")))\nSEEDS = [13, 37, 61]\n\n\ndef save(model, name):\n    path = ROOT / "checkpoints"\n    path.mkdir(exist_ok=True)\n    joblib.dump(model, path / (name + ".joblib"), compress=3)\n\n\ndef phantom(n, seed):\n    rng = np.random.default_rng(seed)\n    y = rng.integers(2, size=n)\n    u, v = np.mgrid[-1:1:28j, -1:1:28j]\n    x = []\n    for label in y:\n        a, b = rng.normal(0, 0.22, 2)\n        scale = rng.uniform(0.14, 0.38)\n        field = np.exp(\n            -(((u - a) / (scale * (1 + 0.55 * label))) ** 2 + ((v - b) / scale) ** 2)\n            / 2\n        )\n        grain = ndi.gaussian_filter(rng.normal(size=u.shape), 0.5 + 0.4 * label)\n        x.append(np.clip(0.30 + field * rng.uniform(0.20, 0.5) + grain * 0.16, 0, 1))\n    return np.asarray(x), y\n\n\ndef image_data(seed, filename):\n    with np.load(DATA / "medmnist" / filename) as raw:\n        return [\n            (raw[k + "_images"].astype("float32") / 255, raw[k + "_labels"].ravel())\n            for k in ["train", "test"]\n        ]\n    return [phantom(n, seed + offset) for n, offset in [(1800, 0), (600, 100)]]\n\n\ndef evaluate(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "balanced_accuracy": float(balanced_accuracy_score(y, p >= 0.5)),\n    }\n\n\ndef shift(x, g, noise):\n    return np.clip(ndi.gaussian_filter(x, (0, 0.9, 0.9)) * g + noise, 0, 1)\n\n\ndef run(seed):\n    (x, y), (z, t) = image_data(seed, "breastmnist.npz")\n    rng = np.random.default_rng(seed)\n    fixed = np.random.default_rng(404)\n    stress = shift(z, 0.68, fixed.normal(0.04, 0.055, z.shape))\n    out = {}\n    for name in ["clean_training", "augmented_training"]:\n        a = x\n        b = y\n        if name == "augmented_training":\n            a = np.concatenate(\n                [\n                    x,\n                    shift(\n                        x,\n                        rng.uniform(0.55, 1.15, (len(x), 1, 1)),\n                        rng.normal(0.03, 0.06, x.shape),\n                    ),\n                ]\n            )\n            b = np.tile(y, 2)\n        model = ExtraTreesClassifier(\n            n_estimators=160,\n            min_samples_leaf=2,\n            max_features=0.2,\n            n_jobs=2,\n            random_state=seed,\n        ).fit(a.reshape(len(a), -1), b)\n        save(model, f"{name}-{seed}")\n        for k, images in [("clean", z), ("shifted", stress)]:\n            out[name + "_" + k] = evaluate(\n                t, model.predict_proba(images.reshape(len(images), -1))[:, 1]\n            )\n    return out\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())